# Sort from a handful

Show it a few examples per label and it sorts the rest. When it isn't sure, it puts the item in "needs a human" instead of guessing. No training run, no API key, runs on CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/hobbylm-embed-cookbook/blob/main/sort-from-a-handful/walkthrough.ipynb)

This notebook runs the recipe end to end: sort 200 real GitHub issues from a handful of examples, fix mistakes with the corrections loop, calibrate the margin, and sort two fictional shared inboxes.

The model is HobbyLM-Embed 140M through Sentence Transformers with `trust_remote_code=True`. Classification uses plain text with no `prompt_name`. Vectors are unit-norm.

## Setup

The cell below sets the repository and branch, and falls back to cloning them when the notebook is not running inside a checkout.

In [1]:
REPO = "fuellabs-ai/hobbylm-embed-cookbook"
BRANCH = "main"
FOLDER = "sort-from-a-handful"

import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

if Path("sort_handful.py").exists():
    print("running from a local checkout")
else:
    if not Path(FOLDER, "sort_handful.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://github.com/{REPO}", FOLDER],
            check=True,
        )
    os.chdir(FOLDER)
print("working folder:", Path.cwd())

running from a local checkout
working folder: /Users/prajhan/Downloads/hobbylm-integrations-check/src/cookbook/sort-from-a-handful


## Install and load the model

Nothing here needs a GPU. On a laptop CPU the model loads in a few seconds.

In [2]:
try:
    import sentence_transformers
    print("sentence-transformers already installed:", sentence_transformers.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    print("installed sentence-transformers")

import time
import numpy as np
from sort_handful import load_model

start = time.perf_counter()
model = load_model()  # harims95/hobbylm-embed-140m, trust_remote_code=True
print(f"model loaded in {time.perf_counter() - start:.1f} s")

sample = model.encode(
    ["fit() crashes with a segmentation fault when n_jobs=-1",
     "How do I get the feature names out of a fitted pipeline?"],
    convert_to_numpy=True,
)
print("dim:", sample.shape[1], "norms:", np.linalg.norm(sample, axis=1).round(3))

W1121 03:40:12.155000 83333 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


sentence-transformers already installed: 6.3.0


model loaded in 2.6 s


dim: 512 norms: [1. 1.]


## Sort 200 real GitHub issues from a handful

`examples/github_issues/items.csv` holds 200 closed issues from [scikit-learn/scikit-learn](https://github.com/scikit-learn/scikit-learn) (BSD-3 licence), 50 per label. The labels are the maintainers' own: `bug`, `documentation`, `question`, `new feature`. They were sampled from the public GitHub API on 21 November 2026: closed issues with the label, no pull requests, a body of at least 80 characters, 50 picked per label with seed 7.

`examples/github_issues/labels.yaml` is a fictional starting point: 5 example texts per label, written by hand. That is the whole setup.

In [3]:
result = subprocess.run(
    [sys.executable, "sort_handful.py", "examples/github_issues/items.csv",
     "--labels", "examples/github_issues/labels.yaml"],
    capture_output=True, text=True,
)
if result.returncode:
    print(result.stderr)
lines = [line for line in result.stdout.splitlines() if line.strip()]
print("\n".join(lines[:13]))
print("... output trimmed; every deferred item is in report.html")

wrote examples/github_issues/sorted.csv
wrote examples/github_issues/report.html
items: 200  margin: 0.040  method: nearest-centroid
sorted: 18  sent to a person: 182
wrongly sorted: 4
  DOC: add explanations about criterion="friedman_mse" in decision trees doc ### Describe the issue linked to the document... -> new feature (expected documentation, score 0.809, margin 0.050)
  HTML repr: keyboard focus is invisible on the expand/collapse toggles (WCAG 2.4.7) > [!WARNING] > This issue is not yet ... -> documentation (expected bug, score 0.732, margin 0.054)
  Kmeans seems to not correctly converge Kmeans seems not to converge to the right equilibrium. ```python random_state = 5... -> bug (expected question, score 0.782, margin 0.052)
  Add collapse option for right-hand "On this page" sidebar > [!WARNING] > This issue is not yet ready for a PR. If you ar... -> documentation (expected new feature, score 0.758, margin 0.053)
sent to a person:
  radius_neighbors with array radius works wit

## What it labelled, what it deferred

At the calibrated margin of 0.04 it labels 18 of 200 and sends 182 to a person, with 4 wrongly sorted. That is what 5 fictional examples buy on real, template-heavy issue text: the sorter stays cautious. The margin sweep and the 5/10/25 table below show what a handful of real examples changes.

In [4]:
import pandas as pd

sorted_csv = pd.read_csv("examples/github_issues/sorted.csv")
wrongly = sorted_csv[(sorted_csv.needs_human == False) & (sorted_csv.label != sorted_csv.expected)]
deferred = sorted_csv[sorted_csv.needs_human == True]
print(f"labelled: {len(sorted_csv) - len(deferred)}  wrongly sorted: {len(wrongly)}  sent to a person: {len(deferred)}")
print()
print("wrongly sorted:")
for _, row in wrongly.iterrows():
    print(f"  got {row.label} (expected {row.expected}) | {row.text[:80]}...")
print()
print("nearest calls among the deferred:")
print(deferred.groupby("label").size().sort_values(ascending=False).head(6).to_string())

labelled: 18  wrongly sorted: 4  sent to a person: 182

wrongly sorted:
  got new feature (expected documentation) | DOC: add explanations about criterion="friedman_mse" in decision trees doc ### D...
  got documentation (expected bug) | HTML repr: keyboard focus is invisible on the expand/collapse toggles (WCAG 2.4....
  got bug (expected question) | Kmeans seems to not correctly converge Kmeans seems not to converge to the right...
  got documentation (expected new feature) | Add collapse option for right-hand "On this page" sidebar > [!WARNING] > This is...

nearest calls among the deferred:
label
new feature      69
bug              53
question         33
documentation    27


## Corrections loop

Move an item to the right label, put it in a CSV with `text` and `label` columns, and pass it with `--corrections`. It is appended as a new example, the centroids are rebuilt, and the run repeats. Three corrections from the four mistakes above:

In [5]:
result = subprocess.run(
    [sys.executable, "sort_handful.py", "examples/github_issues/items.csv",
     "--labels", "examples/github_issues/labels.yaml",
     "--corrections", "examples/github_issues/corrections.csv"],
    capture_output=True, text=True,
)
lines = [line for line in result.stdout.splitlines() if line.strip()]
print("\n".join(lines[:10]))

appending 3 corrections as new examples
wrote examples/github_issues/sorted.csv
wrote examples/github_issues/report.html
items: 200  margin: 0.040  method: nearest-centroid
sorted: 7  sent to a person: 193
wrongly sorted: 2
  Tfidf no genera los cluster correctos para oraciones con poco significado y palabras repetidas ### Describe the workflow... -> documentation (expected new feature, score 0.755, margin 0.040)
  Add collapse option for right-hand "On this page" sidebar > [!WARNING] > This issue is not yet ready for a PR. If you ar... -> documentation (expected new feature, score 0.763, margin 0.043)
sent to a person:
  radius_neighbors with array radius works with n_jobs=1 but raises ValueError for n_jobs>1 ### Describe the bug and give ... -> question or new feature (margin 0.016)


## Calibrate the margin

`calibrate.py` sweeps margins on labelled items, prints wrongly-sorted versus sent-to-human at each, and recommends the smallest margin that keeps wrongly-sorted items at or under one in ten labelled items. It also sorts with 5, 10 and 25 examples per label sampled from the CSV, each time testing only on items that were not used as examples.

In [6]:
result = subprocess.run(
    [sys.executable, "calibrate.py", "examples/github_issues"],
    capture_output=True, text=True,
)
lines = [line for line in result.stdout.splitlines() if line.strip()]
keep = []
skipping = False
for line in lines:
    # show the sweep and the summary, trim the long per-item lists
    if line.startswith("  wrongly-sorted") or line.startswith("  sent-to-human"):
        skipping = True
        continue
    if skipping and not line.startswith("  "):
        skipping = False
    if not skipping:
        keep.append(line)
print("\n".join(keep[:50]))

=== github_issues ===
items: 200 (50 bug, 50 documentation, 50 question, 50 new feature)
margin  labelled  wrongly-sorted  sent-to-human
 0.00      200    97/200          0/200
 0.02       66    20/200          134/200
 0.04       18    4/200          182/200
 0.06        0    0/200          200/200
 0.08        0    0/200          200/200
 0.10        0    0/200          200/200
 0.12        0    0/200          200/200
 0.14        0    0/200          200/200
 0.16        0    0/200          200/200
 0.18        0    0/200          200/200
 0.20        0    0/200          200/200
 0.22        0    0/200          200/200
 0.24        0    0/200          200/200
 0.26        0    0/200          200/200
 0.28        0    0/200          200/200
 0.30        0    0/200          200/200
no margin in the sweep kept wrongly-sorted items to one in ten labelled items. The lowest rate is at margin 0.040 (labelled 18/200, wrongly-sorted 4, sent-to-human 182). A bigger handful of examples, picked 

## The fictional inboxes

Two fictional shared inboxes, written once and not tuned afterwards:

- `examples/careers_inbox`: a careers@ address with referral, direct application, agency CV, interview scheduling and spam. `referral` carries `never_hide: true`, so a referral always reaches a person even when the sorter is sure.
- `examples/school_admissions`: an admissions@ address with fees, documents, visit request, transfer and complaint. `complaint` carries `never_hide: true`.

Accuracy on fictional data mostly measures how clearly the example texts were written.

In [7]:
for folder in ["careers_inbox", "school_admissions"]:
    result = subprocess.run(
        [sys.executable, "sort_handful.py", f"examples/{folder}/inbox.csv",
         "--labels", f"examples/{folder}/labels.yaml"],
        capture_output=True, text=True,
    )
    print("=" * 20, folder, "=" * 20)
    print(result.stdout.strip())

==================== careers_inbox ====================
wrote examples/careers_inbox/sorted.csv
wrote examples/careers_inbox/report.html

items: 30  margin: 0.020  method: nearest-centroid
sorted: 20  sent to a person: 10
wrongly sorted: 2
  My old flatmate Tomasz built the flight controller at AeroWorks and I think he would be great for your firmware opening.... -> direct application (expected referral, score 0.850, margin 0.031)
  I applied for the firmware role two weeks ago and have not heard back. Could you confirm my application arrived and tell... -> direct application (expected interview scheduling, score 0.834, margin 0.020)
sent to a person:
  Hi, this is Marco from the perception team. My former colleague Ilse reviewed our SLAM stack last year and is now lookin... -> referral or direct application (margin 0.063)
  Forwarding on behalf of my brother in law, who has ten years in warehouse automation. He is too modest to write this him... -> direct application or referral (marg

==================== school_admissions ====================
wrote examples/school_admissions/sorted.csv
wrote examples/school_admissions/report.html

items: 30  margin: 0.000  method: nearest-centroid
sorted: 24  sent to a person: 6
wrongly sorted: 0
sent to a person:
  I am writing to complain that the after school club billed us for sessions in October that never ran and nobody has repl... -> complaint or fees (margin 0.036)
  This is the third time this month the school bus has been over 20 minutes late. I need someone to look into it properly. -> complaint or visit request (margin 0.103)
  A teacher told my daughter she was hopeless at maths in front of the class. I want to speak to someone senior about how ... -> complaint or visit request (margin 0.140)
  We were charged twice for the winter concert tickets. The box office says it is the admissions office's problem. Please ... -> complaint or fees (margin 0.033)
  I am not happy that our complaint about the lost uniform was close

## Report

Each run writes `sorted.csv` (text, label, score, margin, needs_human, plus the expected label when the input has one) and `report.html` with the per-label table, the deferred items with their top 2 labels, and every wrong sort. Open the HTML in a browser.

## Limits and where to go next

It sorts by topic and wording, not intent. Labels that differ mostly in intent or tone (a new lead vs a vendor pitch, an agency CV vs spam) will blur, and the margin can be wrong both ways, so set it on your own items and keep a person on anything urgent or sensitive. English only. It labels; it never acts.

The recipe pairs with the What people keep ask for cookbook: first see what people say, name the groups, then sort everything that comes next into them.

## Files

- `sort_handful.py`: the CLI. `python sort_handful.py items.csv --labels labels.yaml`
- `calibrate.py`: the margin sweep and the 5/10/25 table.
- `labels.yaml`: the handful of examples, with optional `never_hide: true`.
- `.github/workflows/label-issues.example.yml`: a GitHub Action that labels new issues on CPU with no secrets beyond the default `GITHUB_TOKEN`.